In [12]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np
import joblib  # Import joblib

# Load the dataset
df = pd.read_csv("Crops_data.csv", encoding="latin1")

# Normalize state names by stripping spaces and converting to lowercase
df["State Name"] = df["State Name"].str.strip().str.lower()
df["Dist Name"] = df["Dist Name"].astype(str).str.strip().str.lower()

# Print available states for debugging
print("Available states in dataset:", df["State Name"].unique())

# Selecting relevant columns
selected_columns = ['State Name', 'Dist Name']
crop_columns = [col for col in df.columns if 'AREA' in col or 'PRODUCTION' in col or 'YIELD' in col]
df_selected = df[selected_columns + crop_columns]

# Encoding categorical variables
label_enc_dist = LabelEncoder()
df_selected['Dist Name'] = label_enc_dist.fit_transform(df_selected['Dist Name'])

# Ensure numerical columns are properly handled
df_selected[crop_columns] = df_selected[crop_columns].apply(pd.to_numeric, errors='coerce')

# Identify best crop based on highest production
if any("PRODUCTION" in col for col in crop_columns):
    production_columns = [col for col in crop_columns if "PRODUCTION" in col]
    df_selected["Best Crop"] = df_selected[production_columns].idxmax(axis=1)
    df_selected["Best Crop"] = df_selected["Best Crop"].str.extract(r'^(.*?) PRODUCTION')
else:
    raise ValueError("No production columns found in the dataset.")

# Handle missing values
df_selected.dropna(inplace=True)

label_enc_crop = LabelEncoder()
df_selected["Best Crop"] = label_enc_crop.fit_transform(df_selected["Best Crop"].fillna("Unknown"))

# Define features and target
X = df_selected.drop(columns=["Best Crop", "State Name", "Dist Name"])
y = df_selected["Best Crop"]

# Ensure all values are numeric
X = X.apply(pd.to_numeric, errors='coerce')
X.fillna(0, inplace=True)

# Store column names to ensure consistent order
feature_columns = list(X.columns)

# Splitting data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# **Save the trained model and label encoder**
joblib.dump(model, "crop_recommendation_model.pkl")
joblib.dump(label_enc_crop, "crop_label_encoder.pkl")
print("Model and label encoder saved successfully.")

# Evaluate model
y_pred = model.predict(X_test)
print(f"Model Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")

# Prediction function
def predict_best_crop():
    state_name = input("Enter State Name: ").strip().lower()
    
    # Print state entered for debugging
    print(f"Searching for: {state_name}")
    
    matching_rows = df_selected[df_selected["State Name"].str.lower().str.strip() == state_name]
    
    if matching_rows.empty:
        print("State not found in dataset. Please check spelling or format.")
        return
    
    # Aggregate features for prediction (taking mean across districts)
    input_features = matching_rows.drop(columns=["Best Crop", "Dist Name", "State Name"]).mean().values.reshape(1, -1)
    
    # Ensure column consistency
    input_df = pd.DataFrame(input_features, columns=X.columns)
    input_df = input_df.reindex(columns=X.columns, fill_value=0)
    
    predicted_crop_encoded = model.predict(input_df)[0]
    predicted_crop = label_enc_crop.inverse_transform([predicted_crop_encoded])[0]
    print(f"Recommended Crop for {state_name.capitalize()}: {predicted_crop}")

# Get user input and predict crop
predict_best_crop()


Available states in dataset: ['chhattisgarh' 'madhya pradesh' 'andhra pradesh' 'telangana' 'karnataka'
 'tamil nadu' 'maharashtra' 'gujarat' 'rajasthan' 'punjab' 'haryana'
 'uttar pradesh' 'uttarakhand' 'assam' 'himachal pradesh' 'kerala'
 'orissa' 'west bengal' 'bihar' 'jharkhand']


C:\Users\vinay\AppData\Local\Temp\ipykernel_17568\1091310404.py:26: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected['Dist Name'] = label_enc_dist.fit_transform(df_selected['Dist Name'])
C:\Users\vinay\AppData\Local\Temp\ipykernel_17568\1091310404.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_selected[crop_columns] = df_selected[crop_columns].apply(pd.to_numeric, errors='coerce')
C:\Users\vinay\AppData\Local\Temp\ipykernel_17568\1091310404.py:34: SettingWithCopyWarning: 
A value is trying

Model and label encoder saved successfully.
Model Accuracy: 92.96%
Searching for: rajasthan
Recommended Crop for Rajasthan: WHEAT
